# Lesson 01 - Giới thiệu về Các tác nhân AI

Chào mừng đến với bài học đầu tiên trong khoá học **AI Agents for Beginners**!

Một **tác nhân AI** là một chương trình sử dụng một mô hình ngôn ngữ lớn (LLM) làm bộ máy suy luận và có thể thực hiện *hành động* trong thế giới thực — gọi API, truy vấn cơ sở dữ liệu, hoặc chạy mã — để đạt được một mục tiêu thay cho người dùng.

Trong sổ tay này, bạn sẽ xây dựng tác nhân đầu tiên của mình: một **Tác nhân Du lịch** đề xuất các điểm đến nghỉ dưỡng. Trong quá trình đó, bạn sẽ học cách:

1. Kết nối với Dịch vụ Tác nhân Azure AI Foundry bằng **Microsoft Agent Framework**.
2. Cung cấp cho tác nhân một **công cụ** — một hàm Python đơn giản mà nó có thể gọi.
3. Chạy tác nhân và kiểm tra phản hồi của nó.
4. Truyền phát phản hồi của tác nhân từng token một.


## Thiết lập

Trước khi chạy notebook này, hãy đảm bảo bạn đã:

1. **Có một dự án Azure AI Foundry** với mô hình trò chuyện đã được triển khai (ví dụ `gpt-4o-mini`).
2. **Đăng nhập bằng Azure CLI** — chạy `az login` trong terminal của bạn.
3. **Đặt các biến môi trường cần thiết:**
   - `AZURE_AI_PROJECT_ENDPOINT` — điểm cuối của dự án Azure AI Foundry của bạn.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — tên mô hình đã được triển khai của bạn.

Ô tếp phía dưới sẽ cài đặt các gói Python bạn cần.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Tạo Đại diện Đầu tiên của Bạn

Một đại diện cần hai thứ:

- **Hướng dẫn** chỉ cho nó *là ai* và *cách hành xử* (một lời nhắc hệ thống).
- **Công cụ** — Các hàm Python được trang trí bằng `@tool` mà đại diện có thể gọi để lấy thông tin hoặc thực hiện các hành động.

Dưới đây chúng tôi định nghĩa một công cụ đơn giản trả về danh sách các điểm đến nghỉ dưỡng phổ biến. Đại diện sẽ sử dụng công cụ này khi người dùng yêu cầu đề xuất du lịch.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming Responses

Để có trải nghiệm tương tác hơn, bạn có thể **stream** phản hồi của agent. Thay vì chờ đợi câu trả lời toàn bộ, agent sẽ gửi từng đoạn văn bản khi chúng được tạo ra. Điều này đặc biệt hữu ích trong các giao diện trò chuyện, nơi bạn muốn hiển thị kết quả ngay lập tức.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Tóm tắt

Trong bài học này bạn đã học cách:

- **Tạo một nhà cung cấp** kết nối với Azure AI Foundry Agent Service qua `AzureAIProjectAgentProvider`.
- **Định nghĩa một công cụ** sử dụng trình trang trí `@tool` để agent có thể gọi các hàm Python của bạn.
- **Chạy agent** với một tin nhắn người dùng và in phản hồi của nó.
- **Phát trực tiếp các phản hồi** để có đầu ra theo thời gian thực.

Trong bài học tiếp theo, chúng ta sẽ khám phá sâu hơn về các framework agentic và học cách cung cấp cho các agent các công cụ mạnh mẽ hơn cùng khả năng suy luận nhiều bước.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố từ chối trách nhiệm**:  
Tài liệu này đã được dịch sử dụng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sự không chính xác. Tài liệu gốc bằng ngôn ngữ gốc của nó nên được coi là nguồn tham khảo chính xác nhất. Đối với thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp bởi con người. Chúng tôi không chịu trách nhiệm về bất kỳ sự hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
